# Transformer Univariate


In this section we implement the Transformer Model using the **TimeSeriesDataset** approach with one-hot encoding.

The Transformer Forecaster is a self-attention-based neural network designed for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. Unlike recurrent models that process sequentially, the Transformer uses attention mechanisms to capture dependencies across the entire sequence simultaneously.

Key Insight: Each training sample represents a single series with its one-hot encoded identifier, allowing the model to learn series-specific patterns while leveraging parallel processing through self-attention mechanisms.

## Architecture

```bash
Input (seq_length, input_size)
    ↓
Input Projection (input_size → d_model)
    ↓
Positional Encoding
    ↓
Transformer Encoder Layers (2 layers)
  ├─ Multi-Head Self-Attention (4 heads)
  ├─ Add & Norm
  ├─ Feedforward (d_model → 256 → d_model)
  └─ Add & Norm
    ↓
Global Average Pooling
    ↓
Dropout
    ↓
Fully Connected (d_model → 1)
    ↓
Output (1 prediction)
```

## Layer Breakdown

- **Input Projection:** Linear layer mapping input features to model dimension (d_model=64)
- **Positional Encoding:** Adds positional information to preserve temporal order
- **Transformer Encoder:** 2 stacked encoder layers with multi-head self-attention (4 heads)
- **Feedforward Network:** 256-dimensional feedforward network within each encoder layer
- **Global Average Pooling:** Aggregates information across all timesteps
- **Dropout:** Applied before final output
- **Output Layer:** Single fully connected layer producing 1-step forecast

## Advantages

- **Parallel Processing:** Processes entire sequence at once (much faster than RNN/LSTM/GRU)
- **Long-Range Dependencies:** Self-attention captures relationships between any two timesteps directly
- **No Gradient Vanishing:** No recurrent connections means no vanishing gradient problem
- **Interpretable:** Attention weights show which timesteps the model focuses on

## Limitations

- **More Parameters:** Requires more data than RNN/LSTM to train effectively
- **Memory Intensive:** Attention mechanism has O(n²) memory complexity with sequence length
- **Positional Encoding Required:** Needs explicit encoding to understand temporal order
- **Short Sequences:** May be overkill for very short sequences (<10 timesteps)

## When to Use

- Dataset is large (>10,000 samples)
- Sequences are moderately long (10-50 timesteps)
- Need to capture complex long-range dependencies
- Training speed is important (parallelizable)
- Interpretability of attention patterns is valuable

## Key Hyperparameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| d_model | 64 | Embedding dimension - higher captures more complexity |
| nhead | 4 | Number of attention heads - allows attending to different aspects |
| num_layers | 2 | Depth of transformer - more layers = more capacity |
| dim_feedforward | 256 | Size of feedforward network within each layer |
| dropout | 0.2 | Dropout rate for regularization |

## Model

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional encoding for Transformer model.
    Adds information about the position of tokens in the sequence.
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, d_model)
        return x + self.pe[:, :x.size(1), :]


class TransformerForecaster(nn.Module):
    """
    Transformer model for MULTIVARIATE time series forecasting.
    Architecture: 
        Input Projection -> Positional Encoding -> 
        Transformer Encoder -> Global Average Pooling -> 
        Dropout -> Fully Connected
    
    Uses self-attention mechanism to capture dependencies.
    Can process entire sequence in parallel (unlike RNN/LSTM).
    """
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, 
                 dim_feedforward=256, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            d_model: Dimension of the model (must be divisible by nhead)
            nhead: Number of attention heads
            num_layers: Number of transformer encoder layers
            dim_feedforward: Dimension of feedforward network
            dropout: Dropout rate
        """
        super(TransformerForecaster, self).__init__()
        
        self.input_size = input_size
        self.d_model = d_model
        
        # Input projection: map input_size to d_model
        self.input_projection = nn.Linear(input_size, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # Important: batch_first=True for (batch, seq, feature) format
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Output layer
        self.fc = nn.Linear(d_model, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # Project input to d_model dimensions
        x = self.input_projection(x)  # (batch_size, seq_length, d_model)
        
        # Add positional encoding
        x = self.pos_encoder(x)  # (batch_size, seq_length, d_model)
        
        # Pass through transformer encoder
        transformer_out = self.transformer_encoder(x)  # (batch_size, seq_length, d_model)
        
        # Global average pooling over sequence dimension
        # Alternative: use last token or first token (like BERT's [CLS])
        pooled = transformer_out.mean(dim=1)  # (batch_size, d_model)
        
        # Apply dropout
        out = self.dropout(pooled)
        
        # Fully connected layer
        out = self.fc(out)  # (batch_size, 1)
        
        return out


## Model Results without Exogenous Features

In this section, the Transformer model is evaluated based on temporal features (value, year, and month) and one-hot encoded series identifiers, without the incorporation of external economic indicators. This baseline approach enables assessment of how well the self-attention mechanism captures series-specific patterns using only historical information and temporal context. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Embedding Dimension | Dim Feedforward | Dropout | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|---------------------:|-----------------------------:|--------:|--------------:|----------:|---------:|
| 0 | 0.35677 | 64 | 32 | 256 | 0.36293 | 0.00022 | 4 | 40.99s |
| 1 | 0.34064 | 128 | 64 | 512 | 0.16428 | 0.00030 | 2 | 47.14s |
| 2 | 0.34070 | 32 | 64 | 256 | 0.14589 | 0.00025 | 2 | 52.97s |





###Best Hyperparameters

Validation Loss: 0.34064

Parameters:
  - learning_rate: 0.00030
  - batch_size: 128
  - d_model: 64
  - nhead: 2
  - num_layers: 1
  - dim_feedforward: 512
  - dropout: 0.16428

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/transformers/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 1 Results](./img/univariate/transformers/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 1 Results](./img/univariate/transformers/fold3/fold_results.png)

### Fold Results
| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 87273.90 | 295.42 | 123.19 | 0.8435 | 73.07% |
| Fold 2 | 72751.26 | 269.72 | 114.57 | 0.8549 | 76.43% |
| Fold 3 | 68857.17 | 262.41 | 105.59 | 0.8689 | 67.63% |
| **Average** | **76294.11 ± 9477.14** | **275.85 ± 17.10** | **114.45 ± 8.99** | **0.8558 ± 0.0119** | **72.37% ± 4.44%** |

### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 11.0% ± 1.6% | 146 |
| 10-20% | 11.4% ± 2.1% | 153 |
| 20-30% | 12.6% ± 2.1% | 168 |
| 30-40% | 10.1% ± 2.1% | 135 |
| >40% | 55.0% ± 5.4% | 733 |

**Comparison with Baseline:**

The Transformer univariate model with one-hot encoding achieves an average SMAPE of **72.37% ± 4.44%**, which is **0.11 percentage points lower** than the baseline 3-month rolling average (72.26% ± 7.06%). This indicates that the Transformer model achieves comparable performance to the baseline while demonstrating greater consistency across folds, as evidenced by the lower standard deviation (4.44% vs 7.06%).


## Model Results with Exogenous Features

In this section, the Transformer model is evaluated based on temporal features (value, year, and month), one-hot encoded series identifiers, and external economic indicators. This approach enables assessment of how well the self-attention mechanism captures series-specific patterns by leveraging both historical information and exogenous economic features. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Embedding Dimension | Dim Feedforward | Dropout | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|---------------------:|-----------------------------:|--------:|--------------:|----------:|---------:|
| 0 | 0.38857 | 32 | 64 | 512 | 0.42190 | 0.00145 | 8 | 03.52s |
| 1 | 0.32203 | 128 | 64 | 128 | 0.23069 | 0.00184 | 4 | 43.24s |
| 2 | 0.32257 | 128 | 64 | 256 | 0.18905 | 0.00030 | 4 | 02.33s |


###Best Hyperparameters

Validation Loss: 0.32203

Parameters:
  - learning_rate: 0.00184
  - batch_size: 128
  - d_model: 64
  - nhead: 4
  - num_layers: 3
  - dim_feedforward: 128
  - dropout: 0.23069

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/transformers_exo/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/univariate/transformers_exo/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 3 Results](./img/univariate/transformers_exo/fold3/fold_results.png)

### Fold Results
| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 84489.99 | 290.67 | 128.24 | 0.8485 | 80.83% |
| Fold 2 | 69839.40 | 264.27 | 117.89 | 0.8607 | 92.13% |
| Fold 3 | 69854.64 | 264.30 | 104.96 | 0.8670 | 75.45% |
| **Average** | **74727.68 ± 8527.60** | **273.08 ± 14.52** | **117.03 ± 11.92** | **0.8587 ± 0.0094** | **82.80% ± 8.34%** |

### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 12.5% ± 2.2% | 167 |
| 10-20% | 10.0% ± 2.0% | 133 |
| 20-30% | 10.6% ± 1.5% | 141 |
| 30-40% | 7.6% ± 1.2% | 101 |
| >40% | 59.3% ± 5.6% | 791 |

**Comparison with Baseline and Univariate Model:**

The Transformer model incorporating exogenous features achieves an average SMAPE of 82.80% ± 8.34%, whereas the univariate Transformer model without exogenous inputs attains a lower SMAPE of 72.37% ± 4.44%. This indicates that the univariate model outperforms the exogenous variant by 10.43 percentage points.

While the univariate Transformer model demonstrates equal performance compared to the baseline 3-month rolling average (72.26% ± 7.06%), the inclusion of exogenous variables causes the model to perform worse than this baseline. The stronger accuracy of the univariate model suggests that temporal dependencies alone are more informative than the combined use of temporal and economic indicators for this dataset. 